In [ ]:
# 1. importing packages
from keras.datasets import fashion_mnist
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import InputLayer, Reshape, Conv2DTranspose, LeakyReLU
from keras.layers import Dense, Conv2D, MaxPooling2D, Dropout, Flatten, Activation
from keras.utils import plot_model
from keras.optimizers import Adam
from keras.models import load_model

# random seed
np.random.seed(67)

In [ ]:
# 2. loading the dataset
(trainX, trainy), (testX, testy) = fashion_mnist.load_data()
print('Train', trainX.shape, trainy.shape)
print('Test', testX.shape, testy.shape)

In [ ]:
# 2. data visualization (first image, inverted image and first 25 images in the same figure)
plt.figure(figsize=(10, 14))

plt.subplot2grid((7, 5), (0, 0), rowspan=2, colspan=2)
plt.imshow(trainX[0], cmap='gray')
plt.title('Grayscale')

plt.subplot2grid((7, 5), (0, 3), rowspan=2, colspan=2)
plt.imshow(trainX[0], cmap='gray_r')
plt.title('Inverted grayscale')

for i in range(25):
    plt.subplot(7, 5, i + 11)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(trainX[i], cmap='gray')
    plt.xlabel(trainy[i])
plt.show()

In [ ]:
# 3. normalizing the data to [0, 1]
trainX = np.expand_dims(trainX, -1)
trainX = trainX / 255.0

In [ ]:
# 4. building the discriminator
def define_discriminator(in_shape=(28, 28, 1)):
    model = Sequential()
    model.add(InputLayer(shape=in_shape))
    model.add(Conv2D(32, (3, 3), strides=(2, 2), padding='same', activation='relu'))
    model.add(Conv2D(64, (3, 3), strides=(2, 2), padding='same', activation='relu'))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))
    # compile model
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    return model

discriminator = define_discriminator()
discriminator.summary()

In [ ]:
# 4. discriminator structure diagram
plot_model(discriminator, show_shapes=True, show_layer_names=True, show_layer_activations=True)

In [ ]:
# 5. building the generator
def define_generator(latent_dim):
    model = Sequential()
    model.add(InputLayer(shape=(latent_dim,)))
    n_nodes = 128 * 7 * 7
    model.add(Dense(n_nodes, activation='relu'))
    model.add(Reshape((7, 7, 128)))
    model.add(Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same', activation='relu'))
    model.add(Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same', activation='relu'))
    model.add(Conv2D(1, (3, 3), strides=(1, 1), padding='same', activation='sigmoid'))
    return model

generator = define_generator(100)
generator.summary()

In [ ]:
# 5. generator structure diagram
plot_model(generator, show_shapes=True, show_layer_names=True, show_layer_activations=True)

In [ ]:
# 6. building the gan
def gan(discriminator, generator):
    # discriminator weights are not trainable inside the gan
    discriminator.trainable = False

    model = Sequential()
    model.add(generator)
    model.add(discriminator)

    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt)
    return model

gan_model = gan(discriminator, generator)
gan_model.summary()

In [ ]:
# 6. gan structure diagram
plot_model(gan_model, show_shapes=True, show_layer_names=True, show_layer_activations=True)

In [ ]:
# 7. training parameters
batch_size = 256
n_epochs = 30
bat_per_epo = int(trainX.shape[0] / batch_size)
half_batch = int(batch_size / 2)
latent_dim = 100

In [ ]:
# 7. helper function to get real samples from the training set
def generate_real_samples(dataset, n_samples):
    ix = np.random.randint(0, dataset.shape[0], n_samples)
    X = dataset[ix]
    y = np.ones((n_samples, 1))
    return X, y

# helper function to get fake samples from the generator
def generate_fake_samples(generator, latent_dim, n_samples):
    x_input = np.random.randn(latent_dim * n_samples)
    x_input = x_input.reshape(n_samples, latent_dim)
    X = generator.predict(x_input, verbose=0)
    y = np.zeros((n_samples, 1))
    return X, y

# helper function to save a plot of generated images
def save_plot(examples, epoch, n=10):
    for i in range(n * n):
        plt.subplot(n, n, 1 + i)
        plt.axis('off')
        plt.imshow(examples[i, :, :, 0], cmap='gray_r')
    filename = 'generated_plot_e%03d.png' % (epoch + 1)
    plt.savefig(filename)
    plt.close()

# helper function to evaluate the model and save the generator
def evaluate(epoch, generator, discriminator, dataset, latent_dim, n_samples=100):
    X_real, y_real = generate_real_samples(dataset, n_samples)
    _, acc_real = discriminator.evaluate(X_real, y_real, verbose=0)

    x_fake, y_fake = generate_fake_samples(generator, latent_dim, n_samples)
    _, acc_fake = discriminator.evaluate(x_fake, y_fake, verbose=0)

    print('>Accuracy real: %.0f%%, fake: %.0f%%' % (acc_real * 100, acc_fake * 100))

    save_plot(x_fake, epoch)
    filename = 'generator_model_%03d.h5' % (epoch + 1)
    generator.save(filename)

In [ ]:
# 7. main training loop
for i in range(n_epochs):

    for j in range(bat_per_epo):

        # train the discriminator on half real and half fake images
        X_real, y_real = generate_real_samples(trainX, half_batch)
        X_fake, y_fake = generate_fake_samples(generator, latent_dim, half_batch)
        X, y = np.vstack((X_real, X_fake)), np.vstack((y_real, y_fake))
        d_loss, _ = discriminator.train_on_batch(X, y)

        # train the generator through the gan with fake images labeled as real
        X_gan = np.random.randn(latent_dim * batch_size)
        X_gan = X_gan.reshape(batch_size, latent_dim)
        y_gan = np.ones((batch_size, 1))
        g_loss = gan_model.train_on_batch(X_gan, y_gan)

    # print once per epoch instead of every batch
    print('>%d/%d, d=%.3f, g=%.3f' % (i + 1, n_epochs, d_loss, g_loss))

    # evaluate every 10 epochs
    if (i + 1) % 10 == 0:
        evaluate(i, generator, discriminator, trainX, latent_dim)

In [ ]:
# 8. loading the final trained generator to make a new image
model = load_model('generator_model_030.h5')

vector = np.random.randn(100 * 1)
vector = vector.reshape(1, latent_dim)

X = model.predict(vector, verbose=0)

plt.imshow(X[0, :, :, 0], cmap='gray_r')
plt.show()